# Practice SPARQL with the Boardgame Knowledge Graph

This notebook loads `bgg_main.ttl` from your local copy of the repository into an in-memory rdflib graph, then lets you practice SPARQL queries against it.

**First-time load takes ~60 seconds** (parsing 28 MB / ~126k triples). After that, all queries run locally — no internet required.

Run the three setup cells first, then work through the practice problems in any order.

## About the data

This notebook works with two TTL files that together form the knowledge graph:

---

### bgg_main.ttl — real data only

Everything in this file comes directly from [BoardGameGeek](https://boardgamegeek.com/). It contains:

- **~33,750 board games** (run the load cell to see the exact triple count)
- The **ontology** (T-Box): classes like `bgg:Game`, `bgg:Creator`, `bgg:Mechanic`, `bgg:Category`, `bgg:PlayerOpinion`; and properties like `bgg:hasName`, `bgg:hasGeekRating`, `bgg:hasCreator`, `bgg:hasMechanic`
- The **game instances** (A-Box): one `bgg:Game` individual per game, e.g. `bgg:174430` is *Gloomhaven*, `bgg:9209` is *Catan*
- Designer, expansion, and reimplementation data

---

### fake_players.ttl — synthetic data, kept separate

BoardGameGeek does not publish individual player data. To practice queries about **ownership** and **personal ratings**, this file contains 200 fictional players with invented game collections and opinions.

It is kept as a separate file — **not merged into bgg_main.ttl** — so the real BGG data stays clean and unmodified. Loading it is opt-in.

---

### Namespace prefixes

| Prefix | URI | Used for |
|--------|-----|---------|
| `bgg:` | `https://raw.githubusercontent.com/susvej/bg_ontology/` | Everything: classes, properties, game instances, mechanic/category/mentalload values |
| `fake:` | `https://vejdemo.se/boardgames/fake#` | Fake player individuals (`fake:AkikoDupont`) and opinion nodes (`fake:AkikoDupont_opinion_174430`) |
| `rdf:` | `http://www.w3.org/1999/02/22-rdf-syntax-ns#` | `rdf:type`, `rdf:Property` |
| `rdfs:` | `http://www.w3.org/2000/01/rdf-schema#` | `rdfs:label`, `rdfs:subClassOf` |
| `owl:` | `http://www.w3.org/2002/07/owl#` | `owl:Class`, `owl:ObjectProperty`, etc. |
| `xsd:` | `http://www.w3.org/2001/XMLSchema#` | Datatypes: `xsd:integer`, `xsd:decimal`, `xsd:string` |

All SPARQL queries in this notebook include these prefix declarations at the top.

In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL
from rdflib.plugins.sparql import prepareQuery
import pandas as pd
import os

print("Imports ok.")

In [ ]:
# Load the graph from the local TTL files (same folder as this notebook)
g = Graph()

print("Loading bgg_main.ttl (~60s)...")
g.parse("../data/bgg_main.ttl", format="ttl")
print(f"  {len(g):,} triples")

print("Loading fake_players.ttl ...")
g.parse("../data/fake_players.ttl", format="ttl")
print(f"  {len(g):,} triples total")

bgg = Namespace("https://raw.githubusercontent.com/susvej/bg_ontology/")
g.namespace_manager.bind("bgg", bgg, override=True)
print("Done.")

In [ ]:
# Explore schema: classes and properties
qname = g.qname

print("CLASSES:")
for s in sorted(g.subjects(RDF.type, OWL.Class), key=str):
    try: print(" -", qname(s))
    except: pass

print("\nOBJECT PROPERTIES:")
for p in sorted(g.subjects(RDF.type, OWL.ObjectProperty), key=str):
    dom = [qname(d) for d in g.objects(p, RDFS.domain)]
    ran = [qname(r) for r in g.objects(p, RDFS.range)]
    print(f"  {qname(p)}  domain={dom}  range={ran}")

print("\nDATATYPE PROPERTIES:")
for p in sorted(g.subjects(RDF.type, OWL.DatatypeProperty), key=str):
    ran = [qname(r) for r in g.objects(p, RDFS.range)]
    print(f"  {qname(p)}  range={ran}")

In [ ]:
# Explore all Category and Mechanic instances
print("MECHANICS:")
for s in sorted(g.subjects(RDF.type, bgg.Mechanic), key=str):
    print(" -", qname(s))

print("\nCATEGORIES:")
for s in sorted(g.subjects(RDF.type, bgg.Category), key=str):
    print(" -", qname(s))

In [ ]:
# Helper: show SPARQL results as a pandas table
def pretty(results):
    data = [{str(k): str(v) for k, v in row.asdict().items()} for row in results]
    return pd.DataFrame(data)

print("pretty() ready.")

---
# Practice problems

Run the cells in order the first time. Some later cells depend on data inserted by earlier ones.

In [ ]:
# P1: Show 10 party games
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P1: Show 10 party games
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name WHERE {
        ?game bgg:hasCategory bgg:PartyGame ;
              bgg:hasName ?name .
    }
    ORDER BY ?name
    LIMIT 10
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P2: 10 games with Geek Rating above 7.0, highest first
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P2: 10 games with Geek Rating above 7.0, highest first
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name ?rating WHERE {
        ?game bgg:hasGeekRating ?rating ;
              bgg:hasName ?name .
        FILTER(?rating > 7.0)
    }
    ORDER BY DESC(?rating)
    LIMIT 10
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P3: Count party games with Geek Rating above 7.0
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P3: Count party games with Geek Rating above 7.0
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (COUNT(?game) AS ?count) WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasGeekRating ?rating ;
              bgg:hasCategory bgg:PartyGame .
        FILTER(?rating > 7)
    }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P4: Games suitable for 3 to 8 players
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P4: Games suitable for 3 to 8 players
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name ?minplay ?maxplay WHERE {
        ?game bgg:hasName ?name ;
              bgg:hasMinPlayers ?minplay ;
              bgg:hasMaxPlayers ?maxplay .
        FILTER(?minplay > 2 && ?maxplay < 8 && ?maxplay > 2)
    }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P5a: Count games for more than 2 players without "pirate" in the title
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P5a: Count games for more than 2 players without "pirate" in the title
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (COUNT(?name) AS ?nonpirategames) WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasName ?name ;
              bgg:hasMaxPlayers ?maxplayers .
        FILTER(?maxplayers > 2 && !regex(?name, "pirate", "i"))
    }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P5b: 5 games with Card Drafting mechanic in Fantasy category, Z to A
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P5b: 5 games with Card Drafting mechanic in Fantasy category, Z to A
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name WHERE {
        ?game bgg:hasName ?name ;
              bgg:hasMechanic bgg:CardDrafting ;
              bgg:hasCategory bgg:Fantasy .
    }
    ORDER BY DESC(?name)
    LIMIT 5
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P6: Card games 2 people can play in under 30 minutes
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P6: Card games 2 people can play in under 30 minutes
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (?name AS ?GameName) (?maxtime AS ?GameTime) WHERE {
        ?game bgg:hasName ?name ;
              bgg:hasMinPlayers 2 ;
              bgg:hasMaxGameTime ?maxtime .
        FILTER(?maxtime <= 30)
    }
    ORDER BY ?name
    LIMIT 10
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P7: Games that do NOT have a playtime specified (FILTER NOT EXISTS)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P7: Games that do NOT have a playtime specified (FILTER NOT EXISTS)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasName ?name .
        FILTER NOT EXISTS { ?game bgg:hasMaxGameTime ?t }
    }
    LIMIT 50
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P8: SELECT games rated above 7.5
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P8: SELECT games rated above 7.5
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name ?rating WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasName ?name ;
              bgg:hasGeekRating ?rating .
        FILTER(?rating > 7.5)
    }
    ORDER BY DESC(?rating)
    LIMIT 20
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P8 INSERT: add bgg:hasQuality bgg:High_Quality to all games rated above 7.5
g.update("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    INSERT {
        # your new triples here
    } WHERE {
        # your pattern here
    }
""")
print("Insert done.")

<details>
<summary>💡 Solution</summary>

```python
# P8 INSERT: add bgg:hasQuality bgg:High_Quality to all games rated above 7.5
g.update("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    INSERT {
        bgg:High_Quality rdf:type rdfs:Class .
        ?game bgg:hasQuality bgg:High_Quality .
    } WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasGeekRating ?rating .
        FILTER(?rating > 7.5)
    }
""")
print("Insert done.")
```

</details>


In [ ]:
# P8 verify: query the newly inserted quality data
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P8 verify: query the newly inserted quality data
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name ?quality WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasName ?name ;
              bgg:hasQuality ?quality .
    }
    LIMIT 20
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P9: MIN and MAX geek rating across all games
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P9: MIN and MAX geek rating across all games
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (MAX(?rating) AS ?maxrating) (MIN(?rating) AS ?minrating) WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasGeekRating ?rating .
    }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P10: Name of the highest AND lowest rated games (subqueries + UNION)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P10: Name of the highest AND lowest rated games (subqueries + UNION)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name ?rating WHERE {
        {
            { SELECT (MAX(?r) AS ?target) WHERE { ?g bgg:hasGeekRating ?r } }
            ?game bgg:hasName ?name ; bgg:hasGeekRating ?rating .
            FILTER(?rating = ?target)
        }
        UNION
        {
            { SELECT (MIN(?r) AS ?target) WHERE { ?g bgg:hasGeekRating ?r } }
            ?game bgg:hasName ?name ; bgg:hasGeekRating ?rating .
            FILTER(?rating = ?target)
        }
    }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P11: BIND a rating label — <=6 "bad", 6-7 "ok", >7 "good"
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P11: BIND a rating label — <=6 "bad", 6-7 "ok", >7 "good"
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name ?rating ?label WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasName ?name ;
              bgg:hasGeekRating ?rating .
        BIND(IF(?rating <= 6, "bad",
              IF(?rating <= 7, "ok", "good")) AS ?label)
    }
    ORDER BY ?rating
    LIMIT 20
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P12: Compare each game to the average play time (subquery + BIND)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P12: Compare each game to the average play time (subquery + BIND)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name ?t ?timerating WHERE {
        { SELECT (AVG(?time) AS ?avgtime) WHERE {
            ?game rdf:type bgg:Game ; bgg:hasMaxGameTime ?time } }
        ?game rdf:type bgg:Game ;
              bgg:hasName ?name ;
              bgg:hasMaxGameTime ?t .
        BIND(IF(?t < ?avgtime, "short", "long") AS ?timerating)
    }
    LIMIT 20
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P13: Count games per max-player count (GROUP BY)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P13: Count games per max-player count (GROUP BY)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (COUNT(?game) AS ?numgames) ?maxplayers WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasMaxPlayers ?maxplayers .
    }
    GROUP BY ?maxplayers
    ORDER BY ASC(?maxplayers)
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P14: 5 games without "war" but with "soldier" in the title (REGEX)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P14: 5 games without "war" but with "soldier" in the title (REGEX)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name WHERE {
        ?game bgg:hasName ?name .
        FILTER(!regex(?name, "war", "i") && regex(?name, "soldier", "i"))
    }
    LIMIT 5
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P15: Average geek rating per category, 2 decimals, descending
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P15: Average geek rating per category, 2 decimals, descending
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (ROUND(AVG(?rating)*100)/100 AS ?avg) (COUNT(?game) AS ?n) ?cat WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasGeekRating ?rating ;
              bgg:hasCategory ?cat .
    }
    GROUP BY ?cat
    ORDER BY DESC(?avg)
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P16: Average playtime for games with exactly 4 max players
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P16: Average playtime for games with exactly 4 max players
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (ROUND(AVG(?time)) AS ?avgtime) WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasMaxPlayers 4 ;
              bgg:hasMaxGameTime ?time .
    }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P17: Average rating per category — only categories with more than 20 games (HAVING)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P17: Average rating per category — only categories with more than 20 games (HAVING)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (AVG(?rating) AS ?avgrating) (COUNT(?game) AS ?n) ?category WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasGeekRating ?rating ;
              bgg:hasCategory ?category .
    }
    GROUP BY ?category
    HAVING (COUNT(?game) > 20)
    ORDER BY ASC(?n)
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P18: Mechanics used by 5+ games, with average and max play time
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P18: Mechanics used by 5+ games, with average and max play time
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?mechanic (ROUND(AVG(?time)) AS ?avgtime) (MAX(?time) AS ?maxtime)
           (COUNT(?game) AS ?n) WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasMechanic ?mechanic ;
              bgg:hasMaxGameTime ?time .
    }
    GROUP BY ?mechanic
    HAVING (COUNT(?game) > 5)
    ORDER BY DESC(?n)
    LIMIT 20
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P19: Creators with more than 5 games AND average rating above 7.0
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P19: Creators with more than 5 games AND average rating above 7.0
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?creator (ROUND(AVG(?rating)*100)/100 AS ?avgrating) (COUNT(?game) AS ?n) WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasCreator ?creator ;
              bgg:hasGeekRating ?rating .
    }
    GROUP BY ?creator
    HAVING (AVG(?rating) > 7 && COUNT(?game) > 5)
    ORDER BY DESC(?avgrating)
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P20: Count games per play-time bucket: <=30, 31-60, >60 min
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P20: Count games per play-time bucket: <=30, 31-60, >60 min
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (COUNT(?game) AS ?n) ?bucket WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasMaxGameTime ?t .
        BIND(IF(?t <= 30, "<=30 min",
              IF(?t <= 60, "31-60 min", ">60 min")) AS ?bucket)
    }
    GROUP BY ?bucket
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P21: Player-count ranges (min+max) with average rating — CONCAT for display
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P21: Player-count ranges (min+max) with average rating — CONCAT for display
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (CONCAT(STR(?min), " to ", STR(?max)) AS ?range)
           (ROUND(AVG(?rating)*100)/100 AS ?avgrating) WHERE {
        ?game rdf:type bgg:Game ;
              bgg:hasMinPlayers ?min ;
              bgg:hasMaxPlayers ?max ;
              bgg:hasGeekRating ?rating .
    }
    GROUP BY ?min ?max
    ORDER BY ?min ?max
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P22: Games with animal names in the title (REGEX alternation)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P22: Games with animal names in the title (REGEX alternation)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name WHERE {
        ?game bgg:hasName ?name .
        FILTER(regex(?name, "(^| )(animal|cat|dog|horse|cow|bear|fox|wolf|penguin)( |$)", "i"))
    }
    ORDER BY ?name
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P23: Count games with no category at all (FILTER NOT EXISTS)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P23: Count games with no category at all (FILTER NOT EXISTS)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (COUNT(?game) AS ?n) WHERE {
        ?game rdf:type bgg:Game .
        FILTER NOT EXISTS { ?game bgg:hasCategory ?cat }
    }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P24: How many games have 1, 2, 3 ... mechanics? (nested GROUP BY)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P24: How many games have 1, 2, 3 ... mechanics? (nested GROUP BY)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT (COUNT(?game) AS ?numgames) ?numMechanics WHERE {
        SELECT DISTINCT ?game (COUNT(?mech) AS ?numMechanics) WHERE {
            ?game rdf:type bgg:Game ;
                  bgg:hasMechanic ?mech .
        }
        GROUP BY ?game
    }
    GROUP BY ?numMechanics
    ORDER BY ?numMechanics
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P25: UNION — games by Reiner Knizia OR in the Abstract_Strategy category
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P25: UNION — games by Reiner Knizia OR in the Abstract_Strategy category
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT DISTINCT ?name WHERE {
        { ?game bgg:hasCreator "Reiner Knizia" ; bgg:hasName ?name }
        UNION
        { ?game bgg:hasCategory bgg:AbstractStrategy ; bgg:hasName ?name }
    }
    LIMIT 10
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P26: XOR — games in Humor OR Party_Game, but NOT both
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P26: XOR — games in Humor OR Party_Game, but NOT both
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name WHERE {
        {   ?game bgg:hasCategory bgg:Humor ; bgg:hasName ?name .
            FILTER NOT EXISTS { ?game bgg:hasCategory bgg:PartyGame } }
        UNION
        {   ?game bgg:hasCategory bgg:PartyGame ; bgg:hasName ?name .
            FILTER NOT EXISTS { ?game bgg:hasCategory bgg:Humor } }
    }
    ORDER BY ?name
    LIMIT 20
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P27: Creators with avg rating + game count, only those with >5 games
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P27: Creators with avg rating + game count, only those with >5 games
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?creator (ROUND(AVG(?rating)*100)/100 AS ?avgrating) (COUNT(?game) AS ?n) WHERE {
        ?game bgg:hasCreator ?creator ;
              bgg:hasGeekRating ?rating .
    }
    GROUP BY ?creator
    HAVING (COUNT(?game) > 5)
    ORDER BY DESC(?n)
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P28: For each game, is its rating above or below the overall average? (subquery + BIND)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P28: For each game, is its rating above or below the overall average? (subquery + BIND)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?name ?rating ?vs WHERE {
        { SELECT (AVG(?r) AS ?avg) WHERE { ?g bgg:hasGeekRating ?r } }
        ?game bgg:hasGeekRating ?rating ;
              bgg:hasName ?name .
        BIND(IF(?rating >= ?avg, "above avg", "below avg") AS ?vs)
    }
    ORDER BY DESC(?rating)
    LIMIT 20
""")
pretty(g.query(myquery))
```

</details>


---
# Property paths and UPDATE

These problems use SPARQL UPDATE (INSERT) and property path expressions (`+`, `^`).

In [ ]:
# P29: INSERT a transitive bgg:expands property with 4 test games
g.update("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    INSERT {
        # your new triples here
    } WHERE {
        # your pattern here
    }
""")

myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P29: INSERT a transitive bgg:expands property with 4 test games
g.update("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    INSERT {
        bgg:expands rdf:type owl:ObjectProperty, owl:TransitiveProperty ;
                    rdfs:domain bgg:Game ; rdfs:range bgg:Game .
        bgg:Game1 rdf:type bgg:Game .
        bgg:Game2 rdf:type bgg:Game ; bgg:expands bgg:Game1 .
        bgg:Game3 rdf:type bgg:Game ; bgg:expands bgg:Game2 .
        bgg:Game4 rdf:type bgg:Game ; bgg:expands bgg:Game3 .
    } WHERE {}
""")

myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?o WHERE { ?s bgg:expands ?o }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P30: Property path + — all (game, expandsTo) pairs, including transitive
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P30: Property path + — all (game, expandsTo) pairs, including transitive
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT DISTINCT ?game ?expandsTo WHERE {
        ?game rdf:type bgg:Game ;
              bgg:expands+ ?expandsTo .
    }
""")
pretty(g.query(myquery))
```

</details>


In [ ]:
# P31: Inverse path ^ — all games that expand Game1 (directly or via chain)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?s ?p ?o {
         ?s ?p ?o
    } LIMIT 5
""")
pretty(g.query(myquery))

<details>
<summary>💡 Solution</summary>

```python
# P31: Inverse path ^ — all games that expand Game1 (directly or via chain)
myquery = prepareQuery("""
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX fake: <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT DISTINCT ?game WHERE {
        bgg:Game1 ^bgg:expands+ ?game .
    }
""")
pretty(g.query(myquery))
```

</details>


---
# OWL modeling exercises

The cells below are **write-your-own** exercises. There is no runnable code — each problem asks you to write an OWL class expression in Turtle syntax.

### OWL 1: Define GameDesigner
A class of individuals that have created at least one Game.

```turtle
bgg:GameDesigner rdf:type owl:Class ;
  owl:equivalentClass [
    rdf:type owl:Restriction ;
    owl:onProperty bgg:hasCreated ;
    owl:someValuesFrom bgg:Game
  ] .
```

### OWL 2: FamilyGame — union of categories
A FamilyGame is any game with category Party_Game OR Trivia.

```turtle
bgg:FamilyGame a owl:Class ;
  owl:equivalentClass [
    owl:unionOf (
      [ a owl:Restriction ; owl:onProperty bgg:hasCategory ; owl:hasValue bgg:PartyGame ]
      [ a owl:Restriction ; owl:onProperty bgg:hasCategory ; owl:hasValue bgg:Trivia ]
    )
  ] .
```

### OWL 3: DiceAndCardGame — intersection of mechanics
Must have BOTH Dice_Rolling AND Card_Drafting.

```turtle
bgg:DiceAndCardGame a owl:Class ;
  owl:equivalentClass [
    owl:intersectionOf (
      [ a owl:Restriction ; owl:onProperty bgg:hasMechanic ; owl:hasValue bgg:DiceRolling ]
      [ a owl:Restriction ; owl:onProperty bgg:hasMechanic ; owl:hasValue bgg:CardDrafting ]
    )
  ] .
```

### OWL 4: KidFriendlyGame — intersection with datatype restriction
Must be in Childrens_Game category AND have max playtime <= 30 minutes.

```turtle
bgg:KidFriendlyGame a owl:Class ;
  owl:equivalentClass [
    owl:intersectionOf (
      [ a owl:Restriction ; owl:onProperty bgg:hasCategory ; owl:hasValue bgg:ChildrensGame ]
      [ a owl:Restriction ; owl:onProperty bgg:hasMaxGameTime ;
        owl:allValuesFrom [ a rdfs:Datatype ; owl:onDatatype xsd:integer ;
          owl:withRestrictions ( [ xsd:maxInclusive 30 ] ) ] ]
    )
  ] .
```

### OWL 5: WarGame / PeaceGame — complement
WarGame has a conflict-based category. PeaceGame is its complement.

```turtle
bgg:WarGame a owl:Class ;
  owl:equivalentClass [
    a owl:Restriction ;
    owl:onProperty bgg:hasCategory ;
    owl:someValuesFrom [ a owl:Class ; owl:oneOf (bgg:WorldWarI bgg:WorldWarII bgg:Wargame) ]
  ] .

bgg:PeaceGame a owl:Class ;
  owl:complementOf bgg:WarGame .
```

### OWL 6: Transitive property — game expansions
Create a property that means "this game is an expansion of another", transitively (if A expands B and B expands C, then A expands C).

```turtle
bgg:expands a owl:ObjectProperty, owl:TransitiveProperty ;
  rdfs:domain bgg:Game ;
  rdfs:range  bgg:Game .
```

Then run P29–P31 above to see this in action with SPARQL property paths.